<a href="https://colab.research.google.com/github/apsarak26/capstone_project_CS5998/blob/main/Pilot_Model_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from google.colab import drive
drive.mount('/content/drive')

dataset_path = (
    "/content/drive/MyDrive/integrated_processed/"
    "aeolus_faers_2014q1_integrated_pilot.csv"
)

df = pd.read_csv(
    dataset_path,
    dtype={
        "report_id": str,
        "primaryid": str,
        "caseid": str,
        "standard_concept_id": str,
        "drug_seq": str
    },
    low_memory=False
)

print("Dataset shape:", df.shape)
print("Unique reports:", df["report_id"].nunique())

display(df.head())

Mounted at /content/drive
Dataset shape: (251245, 49)
Unique reports: 251245


,report_id,standard_concept_id,role_cod,provisional_hypersensitivity_label,primaryid,caseid,caseversion,age_years,age_missing,gender_clean,...,indication_terms,indication_record_available,usable_indication_available,therapy_record_count,earliest_therapy_start,latest_therapy_end,max_calculated_duration_days,max_reported_duration_days,multiple_therapy_records,therapy_record_available
0,PRIMARY_92198372,1110942,PS,0,92198372,9219837,2,NaN,1,Female,...,ASTHMA,1,1,1.0,2011-05-26,NaN,NaN,NaN,0.0,1
1,PRIMARY_98124391,735843,PS,0,98124391,9812439,1,NaN,1,Female,...,MULTIPLE SCLEROSIS,1,1,1.0,2013-12-17,NaN,NaN,NaN,0.0,1
2,PRIMARY_98164081,917336,PS,0,98164081,9816408,1,18.0,0,Male,...,NaN,0,0,1.0,2013-08-09,NaN,NaN,NaN,0.0,1
3,PRIMARY_98972741,1397141,PS,0,98972741,9897274,1,NaN,1,Unknown,...,GLIOBLASTOMA MULTIFORME,1,1,NaN,NaN,NaN,NaN,NaN,NaN,0
4,PRIMARY_100338831,723013,PS,0,100338831,10033883,1,50.0,0,Male,...,NaN,1,0,NaN,NaN,NaN,NaN,NaN,NaN,0


Define feature groups

In [2]:
target_col = "provisional_hypersensitivity_label"

# Core features
core_features = [
    "standard_concept_id",
    "age_years",
    "gender_clean",
    "route_clean"
]

# Additional drug features
drug_features = [
    "dose_form_clean",
    "dose_freq_clean",
    "dose_mg"
]

# Medication burden features
medication_features = [
    "total_drug_count",
    "concomitant_drug_count",
    "high_medication_burden"
]

# Treatment indication features
indication_features = [
    "indication_terms"
]

# Optional demographic features
optional_demographic_features = [
    "weight_kg",
    "occr_country_clean"
]

# Missingness features for separate experiments
missingness_features = [
    "age_missing",
    "gender_missing",
    "weight_missing",
    "route_missing",
    "dose_form_missing",
    "dose_missing",
    "dose_freq_missing"
]

# Identifiers retained for tracking and splitting
identifier_columns = [
    "report_id",
    "primaryid",
    "caseid"
]

feature_groups = {
    "Core": core_features,
    "Drug": drug_features,
    "Medication": medication_features,
    "Indication": indication_features,
    "Optional demographics": optional_demographic_features,
    "Missingness": missingness_features
}

for group, columns in feature_groups.items():
    missing = [col for col in columns if col not in df.columns]

    print(f"{group}: {len(columns)} features")
    print("Missing columns:", missing)
    print()

Core: 4 features
Missing columns: []

Drug: 3 features
Missing columns: []

Medication: 3 features
Missing columns: []

Indication: 1 features
Missing columns: []

Optional demographics: 2 features
Missing columns: []

Missingness: 7 features
Missing columns: []



Final data quality assessment

Validate numerical features

In [3]:
numerical_features = [
    "age_years",
    "weight_kg",
    "dose_mg",
    "total_drug_count",
    "concomitant_drug_count",
    "secondary_suspect_count",
    "interacting_drug_count"
]

display(
    df[numerical_features].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
)

,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
age_years,161526.0,53.625796,18.997507,0.000000,4.000,41.0,56.0,67.0,82.0,89.0,104.00
weight_kg,54501.0,76.171659,24.872953,0.081200,4.260,61.0,74.0,89.8,118.0,150.0,308.72
dose_mg,134220.0,248.475224,7085.519366,0.000006,0.018,10.0,40.0,120.0,600.0,2000.0,2000000.00
total_drug_count,251245.0,3.475225,4.450968,1.000000,1.000,1.0,2.0,4.0,12.0,21.0,204.00
concomitant_drug_count,251245.0,1.776270,3.948308,0.000000,0.000,0.0,0.0,2.0,9.0,17.0,184.00
secondary_suspect_count,251245.0,0.687456,1.645474,0.000000,0.000,0.0,0.0,1.0,3.0,7.0,81.00
interacting_drug_count,251245.0,0.011495,0.242730,0.000000,0.000,0.0,0.0,0.0,0.0,0.0,35.00


Investigate extreme dosage values

In [4]:
print("Dosage statistics:")

display(
    df["dose_mg"].describe(
        percentiles=[0.90, 0.95, 0.99, 0.999]
    )
)

print("\nTop 20 highest dosage records:")

display(
    df.nlargest(20, "dose_mg")[
        [
            "standard_concept_id",
            "drugname_clean",
            "dose_mg",
            "dose_unit_clean",
            "route_clean",
            "dose_freq_clean",
            target_col
        ]
    ]
)

print("\nNon-positive dosage values:",
      (df["dose_mg"] <= 0).sum())

Dosage statistics:


,dose_mg
count,1.342200e+05
mean,2.484752e+02
std,7.085519e+03
min,6.000000e-06
50%,4.000000e+01
90%,3.000000e+02
95%,6.000000e+02
99%,2.000000e+03
99.9%,2.500000e+04
max,2.000000e+06



Top 20 highest dosage records:


,standard_concept_id,drugname_clean,dose_mg,dose_unit_clean,route_clean,dose_freq_clean,provisional_hypersensitivity_label
245350,1503297,METFORMIN (UNKNOWN),2000000.0,G,ORAL,NaN,0
116264,968426,MESAVANCOL,1200000.0,G,ORAL,BID,0
143099,1792515,CHLOROQUINE,548000.0,GM,NaN,NaN,0
180223,937368,REMICADE,500000.0,GM,INTRAVENOUS,NaN,0
202787,1792515,CHLOROQUINE,274000.0,GM,NaN,NaN,0
187079,40241937,ZELBORAF,240000.0,G,ORAL,BID,0
207565,43526424,TECFIDERA,240000.0,G,ORAL,BID,0
67446,986417,MIRALAX,238000.0,G,ORAL,1X,0
134406,986417,MIRALAX,238000.0,G,ORAL,NaN,0
75327,19117912,GAMMAGARD LIQUID,180000.0,G,INTRAVENOUS,NaN,0



Non-positive dosage values: 0


Assess categorical features

Categorical feature summary

In [5]:
categorical_features = [
    "standard_concept_id",
    "gender_clean",
    "route_clean",
    "dose_form_clean",
    "dose_freq_clean",
    "indication_terms"
]

categorical_summary = pd.DataFrame({
    "unique_values": df[categorical_features].nunique(),
    "missing_count": df[categorical_features].isna().sum(),
    "missing_percentage": (
        df[categorical_features].isna().mean() * 100
    ).round(2)
})

display(categorical_summary)

for col in categorical_features:
    print(f"\nTop 10 values: {col}")
    display(df[col].value_counts(dropna=False).head(10))

,unique_values,missing_count,missing_percentage
standard_concept_id,1331,0,0.00
gender_clean,3,0,0.00
route_clean,54,90634,36.07
dose_form_clean,227,78666,31.31
dose_freq_clean,67,137274,54.64
indication_terms,4024,66022,26.28



Top 10 values: standard_concept_id


,count
standard_concept_id,
1119119,37973
1151789,11686
1337068,11009
19026972,7435
43526424,5964
19092849,5517
735843,5336
40170680,4675
722424,4422



Top 10 values: gender_clean


,count
gender_clean,
Female,150147
Male,84060
Unknown,17038



Top 10 values: route_clean


... (3 outputs left out)


Top 10 values: dose_freq_clean


,count
dose_freq_clean,
NaN,137274
QD,37313
QOW,25666
BID,19353
/WK,14541
/MONTH,6298
TID,1937
BIW,1578
1X,1454



Top 10 values: indication_terms


,count
indication_terms,
NaN,66022
RHEUMATOID ARTHRITIS,20222
MULTIPLE SCLEROSIS,16461
CROHN'S DISEASE,10470
PSORIASIS,7946
PLASMA CELL MYELOMA,6905
PERITONEAL DIALYSIS,5221
PSORIATIC ARTHROPATHY,5087
CONTRACEPTION,3505


Assess rare drug concepts

In [6]:
drug_counts = df["standard_concept_id"].value_counts()

print("Total unique drug concepts:", len(drug_counts))

for threshold in [5, 10, 20, 50, 100]:

    rare_drugs = drug_counts[drug_counts < threshold]

    print(f"\nDrug concepts with fewer than {threshold} reports:")
    print("Number of drug concepts:", len(rare_drugs))
    print("Number of affected reports:", rare_drugs.sum())
    print(
        "Percentage of dataset:",
        round(rare_drugs.sum() / len(df) * 100, 2)
    )

Total unique drug concepts: 1331

Drug concepts with fewer than 5 reports:
Number of drug concepts: 430
Number of affected reports: 817
Percentage of dataset: 0.33

Drug concepts with fewer than 10 reports:
Number of drug concepts: 588
Number of affected reports: 1881
Percentage of dataset: 0.75

Drug concepts with fewer than 20 reports:
Number of drug concepts: 733
Number of affected reports: 3940
Percentage of dataset: 1.57

Drug concepts with fewer than 50 reports:
Number of drug concepts: 901
Number of affected reports: 9252
Percentage of dataset: 3.68

Drug concepts with fewer than 100 reports:
Number of drug concepts: 1034
Number of affected reports: 18619
Percentage of dataset: 7.41


Define final modelling feature sets

In [7]:
# Experiment 1: Basic features
features_e1 = [
    "standard_concept_id",
    "age_years",
    "gender_clean"
]

# Experiment 2: Extended drug features
features_e2 = features_e1 + [
    "route_clean",
    "dose_form_clean",
    "dose_freq_clean"
]

# Experiment 3: Contextual features
features_e3 = features_e2 + [
    "total_drug_count",
    "concomitant_drug_count",
    "indication_terms"
]

experiments = {
    "E1_Basic": features_e1,
    "E2_Extended": features_e2,
    "E3_Contextual": features_e3
}

for name, features in experiments.items():
    print(f"\n{name}")
    print("Number of raw features:", len(features))

    for feature in features:
        print(" -", feature)

    assert all(col in df.columns for col in features)


E1_Basic
Number of raw features: 3
 - standard_concept_id
 - age_years
 - gender_clean

E2_Extended
Number of raw features: 6
 - standard_concept_id
 - age_years
 - gender_clean
 - route_clean
 - dose_form_clean
 - dose_freq_clean

E3_Contextual
Number of raw features: 9
 - standard_concept_id
 - age_years
 - gender_clean
 - route_clean
 - dose_form_clean
 - dose_freq_clean
 - total_drug_count
 - concomitant_drug_count
 - indication_terms


Create the modelling dataset

In [8]:
tracking_columns = [
    "report_id",
    "primaryid",
    "caseid"
]

model_columns = (
    tracking_columns
    + features_e3
    + [target_col]
)

model_df = df[model_columns].copy()

print("Modelling dataset shape:", model_df.shape)

print("\nDuplicate report IDs:",
      model_df["report_id"].duplicated().sum())

print("Duplicate case IDs:",
      model_df["caseid"].duplicated().sum())

print("Missing target values:",
      model_df[target_col].isna().sum())

print("\nTarget distribution:")
display(model_df[target_col].value_counts())

display(model_df.head())

Modelling dataset shape: (251245, 13)

Duplicate report IDs: 0
Duplicate case IDs: 0
Missing target values: 0

Target distribution:


,count
provisional_hypersensitivity_label,
0,245919
1,5326


,report_id,primaryid,caseid,standard_concept_id,age_years,gender_clean,route_clean,dose_form_clean,dose_freq_clean,total_drug_count,concomitant_drug_count,indication_terms,provisional_hypersensitivity_label
0,PRIMARY_92198372,92198372,9219837,1110942,NaN,Female,SUBCUTANEOUS,SOLUTION FOR INJECTION,NaN,22,18,ASTHMA,0
1,PRIMARY_98124391,98124391,9812439,735843,NaN,Female,INTRAVENOUS,INTRAVENOUS INFUSION,/MONTH,1,0,MULTIPLE SCLEROSIS,0
2,PRIMARY_98164081,98164081,9816408,917336,18.0,Male,TOPICAL,NaN,NaN,1,0,NaN,0
3,PRIMARY_98972741,98972741,9897274,1397141,NaN,Unknown,NaN,NaN,QOW,4,0,GLIOBLASTOMA MULTIFORME,0
4,PRIMARY_100338831,100338831,10033883,723013,50.0,Male,ORAL,NaN,NaN,3,0,NaN,0


Split training, validation and test datasets

In [9]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Confirm one report per case
assert model_df["caseid"].is_unique
assert model_df["caseid"].notna().all()

# 70% training, 30% temporary
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE
)

# Divide remaining 30% equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[target_col],
    random_state=RANDOM_STATE
)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Testing dataset:", test_df.shape)

Training dataset: (175871, 13)
Validation dataset: (37687, 13)
Testing dataset: (37687, 13)


In [10]:
splits = {
    "Training": train_df,
    "Validation": val_df,
    "Testing": test_df
}

split_summary = []

for name, data in splits.items():

    positives = int(data[target_col].sum())
    negatives = len(data) - positives

    split_summary.append({
        "Dataset": name,
        "Total reports": len(data),
        "Negative": negatives,
        "Positive": positives,
        "Positive percentage": round(
            positives / len(data) * 100, 4
        ),
        "Dataset percentage": round(
            len(data) / len(model_df) * 100, 2
        )
    })

split_summary_df = pd.DataFrame(split_summary)

display(split_summary_df)

,Dataset,Total reports,Negative,Positive,Positive percentage,Dataset percentage
0,Training,175871,172143,3728,2.1197,70.0
1,Validation,37687,36888,799,2.1201,15.0
2,Testing,37687,36888,799,2.1201,15.0


In [11]:
train_cases = set(train_df["caseid"])
val_cases = set(val_df["caseid"])
test_cases = set(test_df["caseid"])

print("Train-validation overlap:",
      len(train_cases & val_cases))

print("Train-test overlap:",
      len(train_cases & test_cases))

print("Validation-test overlap:",
      len(val_cases & test_cases))

assert not train_cases.intersection(val_cases)
assert not train_cases.intersection(test_cases)
assert not val_cases.intersection(test_cases)

assert (
    len(train_df) + len(val_df) + len(test_df)
    == len(model_df)
)

assert (
    train_df[target_col].sum()
    + val_df[target_col].sum()
    + test_df[target_col].sum()
    == model_df[target_col].sum()
)

print("\nAll split integrity checks passed.")

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

All split integrity checks passed.


Preprocessing pipeline Preparation

In [12]:
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    MultiLabelBinarizer
)

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from scipy import sparse

import joblib
import json

In [13]:
def create_preprocessor(features):

    transformers = []

    # Drug concept encoding
    if "standard_concept_id" in features:

        drug_encoder = OneHotEncoder(
            min_frequency=10,
            handle_unknown="infrequent_if_exist",
            sparse_output=True,
            dtype=np.float32
        )

        transformers.append(
            (
                "drug",
                drug_encoder,
                ["standard_concept_id"]
            )
        )

    # Numerical features
    numerical_cols = [
        col for col in [
            "age_years",
            "total_drug_count",
            "concomitant_drug_count"
        ]
        if col in features
    ]

    if numerical_cols:

        numerical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_cols
            )
        )

    # Other categorical features
    categorical_cols = [
        col for col in [
            "gender_clean",
            "route_clean",
            "dose_form_clean",
            "dose_freq_clean"
        ]
        if col in features
    ]

    if categorical_cols:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="MISSING"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32
                )
            )
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_cols
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0
    )

In [14]:
from collections import Counter

def parse_indications(series):

    return [
        sorted(set(
            term.strip()
            for term in value.split("|")
            if term.strip()
        ))
        for value in series.fillna("").astype(str)
    ]

train_indications = parse_indications(
    train_df["indication_terms"]
)

val_indications = parse_indications(
    val_df["indication_terms"]
)

test_indications = parse_indications(
    test_df["indication_terms"]
)

# Learn indication frequencies from training data only
indication_counts = Counter(
    term
    for terms in train_indications
    for term in terms
)

# Retain indications appearing in at least 10 training reports
retained_indications = sorted([
    term
    for term, count in indication_counts.items()
    if count >= 10
])

retained_set = set(retained_indications)

def filter_indications(records):

    return [
        [term for term in terms if term in retained_set]
        for terms in records
    ]

indication_encoder = MultiLabelBinarizer(
    classes=retained_indications,
    sparse_output=True
)

X_train_indi = indication_encoder.fit_transform(
    filter_indications(train_indications)
)

X_val_indi = indication_encoder.transform(
    filter_indications(val_indications)
)

X_test_indi = indication_encoder.transform(
    filter_indications(test_indications)
)

print("Unique training indications:",
      len(indication_counts))

print("Retained indication features:",
      len(retained_indications))

print("Training indication matrix:", X_train_indi.shape)
print("Validation indication matrix:", X_val_indi.shape)
print("Testing indication matrix:", X_test_indi.shape)

Unique training indications: 2856
Retained indication features: 658
Training indication matrix: (175871, 658)
Validation indication matrix: (37687, 658)
Testing indication matrix: (37687, 658)


In [15]:
processed_experiments = {}
preprocessors = {}

for experiment_name, features in experiments.items():

    print(f"\nProcessing {experiment_name}")

    preprocessor = create_preprocessor(features)

    X_train = preprocessor.fit_transform(
        train_df[features]
    )

    X_val = preprocessor.transform(
        val_df[features]
    )

    X_test = preprocessor.transform(
        test_df[features]
    )

    feature_names = list(
        preprocessor.get_feature_names_out()
    )

    # Add multi-hot indication features for E3
    if "indication_terms" in features:

        X_train = sparse.hstack(
            [X_train, X_train_indi],
            format="csr"
        )

        X_val = sparse.hstack(
            [X_val, X_val_indi],
            format="csr"
        )

        X_test = sparse.hstack(
            [X_test, X_test_indi],
            format="csr"
        )

        feature_names += [
            f"indication__{term}"
            for term in indication_encoder.classes_
        ]

    X_train = sparse.csr_matrix(
        X_train, dtype=np.float32
    )

    X_val = sparse.csr_matrix(
        X_val, dtype=np.float32
    )

    X_test = sparse.csr_matrix(
        X_test, dtype=np.float32
    )

    processed_experiments[experiment_name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "feature_names": feature_names
    }

    preprocessors[experiment_name] = preprocessor

    print("Training shape:", X_train.shape)
    print("Validation shape:", X_val.shape)
    print("Testing shape:", X_test.shape)
    print("Encoded features:", len(feature_names))


Processing E1_Basic
Training shape: (175871, 677)
Validation shape: (37687, 677)
Testing shape: (37687, 677)
Encoded features: 677

Processing E2_Extended
Training shape: (175871, 1004)
Validation shape: (37687, 1004)
Testing shape: (37687, 1004)
Encoded features: 1004

Processing E3_Contextual
Training shape: (175871, 1664)
Validation shape: (37687, 1664)
Testing shape: (37687, 1664)
Encoded features: 1664


In [16]:
y_train = train_df[target_col].to_numpy(dtype=np.int32)
y_val = val_df[target_col].to_numpy(dtype=np.int32)
y_test = test_df[target_col].to_numpy(dtype=np.int32)

print("Training target:", y_train.shape)
print("Validation target:", y_val.shape)
print("Testing target:", y_test.shape)

# Calculate class imbalance ratio from training data
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Training negatives:", negative_count)
print("Training positives:", positive_count)

print(
    "Negative-to-positive ratio:",
    round(scale_pos_weight, 4)
)

Training target: (175871,)
Validation target: (37687,)
Testing target: (37687,)
Training negatives: 172143
Training positives: 3728
Negative-to-positive ratio: 46.1757


In [17]:
model_output_dir = (
    "/content/drive/MyDrive/model_processed/pilot_2014q1"
)

os.makedirs(model_output_dir, exist_ok=True)

for folder in [
    "splits",
    "targets",
    "encoders",
    *experiments.keys()
]:
    os.makedirs(
        os.path.join(model_output_dir, folder),
        exist_ok=True
    )

print("Output directories created.")

Output directories created.


In [18]:
for name, data in processed_experiments.items():

    experiment_dir = os.path.join(
        model_output_dir, name
    )

    for split in ["train", "val", "test"]:

        sparse.save_npz(
            os.path.join(
                experiment_dir,
                f"X_{split}.npz"
            ),
            data[f"X_{split}"],
            compressed=True
        )

    with open(
        os.path.join(experiment_dir, "feature_names.json"),
        "w"
    ) as f:
        json.dump(data["feature_names"], f, indent=2)

    print(f"{name} saved successfully.")

E1_Basic saved successfully.
E2_Extended saved successfully.
E3_Contextual saved successfully.


In [19]:
targets_dir = os.path.join(
    model_output_dir, "targets"
)

np.save(os.path.join(targets_dir, "y_train.npy"), y_train)
np.save(os.path.join(targets_dir, "y_val.npy"), y_val)
np.save(os.path.join(targets_dir, "y_test.npy"), y_test)

splits_dir = os.path.join(
    model_output_dir, "splits"
)

train_df.to_csv(
    os.path.join(splits_dir, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(splits_dir, "validation.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(splits_dir, "test.csv"),
    index=False
)

print("Target arrays and original splits saved.")

Target arrays and original splits saved.


In [20]:
encoders_dir = os.path.join(
    model_output_dir, "encoders"
)

for name, preprocessor in preprocessors.items():

    joblib.dump(
        preprocessor,
        os.path.join(
            encoders_dir,
            f"{name}_preprocessor.joblib"
        )
    )

joblib.dump(
    indication_encoder,
    os.path.join(
        encoders_dir,
        "indication_encoder.joblib"
    )
)

print("All preprocessing objects saved.")

All preprocessing objects saved.


Save experiment metadata

In [22]:
import sklearn
import scipy
import sys

metadata = {
    "dataset": "AEOLUS_FAERS_2014Q1_Pilot",
    "total_reports": int(len(model_df)),
    "target_column": target_col,
    "target_definition": "Provisional report-level hypersensitivity label",
    "random_state": RANDOM_STATE,

    "split_ratios": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },

    "split_sizes": {
        "train": int(len(train_df)),
        "validation": int(len(val_df)),
        "test": int(len(test_df))
    },

    "training_class_distribution": {
        "negative": int(negative_count),
        "positive": int(positive_count)
    },

    "scale_pos_weight": float(scale_pos_weight),

    "minimum_drug_frequency": 10,
    "minimum_indication_frequency": 10,

    "retained_indication_features": int(
        len(retained_indications)
    ),

    "experiments": experiments,

    "encoded_feature_counts": {
        name: int(data["X_train"].shape[1])
        for name, data in processed_experiments.items()
    },

    "library_versions": {
        "python": sys.version,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__
    }
}

with open(
    os.path.join(model_output_dir, "metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=4)

split_summary_df.to_csv(
    os.path.join(
        model_output_dir,
        "split_summary.csv"
    ),
    index=False
)

print("Experiment metadata and summaries saved.")

Experiment metadata and summaries saved.


Verify saved files

In [23]:
for name, data in processed_experiments.items():

    for split in ["train", "val", "test"]:

        loaded = sparse.load_npz(
            os.path.join(
                model_output_dir,
                name,
                f"X_{split}.npz"
            )
        )

        original = data[f"X_{split}"]

        assert loaded.shape == original.shape
        assert (loaded != original).nnz == 0

    print(f"{name}: Saved matrices verified.")

for split, original in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test)
]:

    loaded = np.load(
        os.path.join(
            targets_dir,
            f"y_{split}.npy"
        )
    )

    assert np.array_equal(loaded, original)

print("\nAll saved target arrays verified.")

total_size = sum(
    os.path.getsize(os.path.join(root, file))
    for root, dirs, files in os.walk(model_output_dir)
    for file in files
)

print(
    "\nTotal saved file size:",
    round(total_size / (1024 ** 2), 2),
    "MB"
)

print("\nModel preprocessing completed successfully.")

E1_Basic: Saved matrices verified.
E2_Extended: Saved matrices verified.
E3_Contextual: Saved matrices verified.

All saved target arrays verified.

Total saved file size: 29.33 MB

Model preprocessing completed successfully.
